<!-- nav -->
[← 5 · Deciding what you can now](05_Deciding_What_You_Can_Now.ipynb) · [Home](../../README.md) · [7 · One constraint, many languages →](07_One_Constraint_Many_Languages.ipynb)

# 6 · Analyzing and rewriting constraints

**The problem.** The QA team has hundreds of stored constraints. Before a schema change, they need to know which
constraints read the property being renamed. For reviews, they want every constraint written the same way, with the
property first. Both are programs over constraints: one reads them, the other rewrites them.

Because a constraint is data, those programs are ordinary code over a small tree. This case study writes both, and shows
the tools that make them short: `walk`, `fold`, `same`, and rewriting constraints.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Symbolics, Terms } from "@mbse/expressions/Framework";
import { Hole, Pairwise, Pattern, Rule } from "@mbse/expressions/Framework/Translators";
import { cold_chain, show } from "./toolkit.js";

const store = cold_chain();
const Expression = store.Expressions.Framework.Expression;
const [shipment, c] = [E.variable("shipment"), E.variable("c")];
const celsius = shipment.celsius;
const writers: Record<string, any> = {
  in_range: E.literal(2.0).le(celsius).and_(celsius.le(8.0)),
  on_time: shipment.hours.add(E.variable("to_go")).le(72n),
  allowed: shipment.readings.count_where("c", c.gt(8.0)).le(2n).and_(shipment.readings.max().le(12.0)),
  reported: shipment.has("celsius"),
};
const constraints: Record<string, any> = store.transaction(() => Object.fromEntries(
  Object.entries(writers).map(([name, writer]) => [name, Expression((e: any) => e.root(writer))])));

## Step 1: a constraint is a tree of terms

Every term has a **form**, which `Terms.form(term)` reads: its `kind` (literal, variable, call, let, quantifier), its
`attributes` (a literal's value, a call's callable, by name) and its ordered `arguments`, which are terms.
`Terms.walk` visits each term within a constraint, parents before their arguments:

In [2]:
for (const term of Terms.walk(constraints.in_range)) {
  const form = Terms.form(term);
  console.log(form.kind.padEnd(10), Object.fromEntries(form.attributes), form.arguments.length);
}

call       { callable: "and" } 2


call       { callable: "le" } 2


literal    { value: 2 } 0


call       { callable: "get" } 2


variable   { name: "shipment" } 0


...

call       { callable: "le" } 2


call       { callable: "get" } 2


variable   { name: "shipment" } 0


literal    { value: "celsius" } 0


literal    { value: 8 } 0


`get(shipment, 'celsius')` appears twice in the walk, as the constraint uses it twice: a term is a value, not an object of
its own, so each use is a term of its own, as it was saved at each use in case study 1.

## Step 2: which constraints read a property

A call of `get` or `has` whose second argument is a literal names the property it reads. Collecting those is a loop over the
walk:

In [3]:
function reads(constraint: any): string[] {
  const found = new Set<string>();
  for (const term of Terms.walk(constraint)) {
    const form = Terms.form(term);
    if (form.kind === "call" && ["get", "has"].includes(form.attributes.get("callable") as string)) {
      const name = Terms.form(form.arguments[1]);
      if (name.kind === "literal") found.add(name.attributes.get("value") as string);
    }
  }
  return [...found].sort();
}

console.log(Object.fromEntries(Object.entries(constraints).map(([name, constraint]) => [name, reads(constraint)])));

{
  in_range: [ "celsius" ],
  on_time: [ "hours" ],
  allowed: [ "readings" ],
  reported: [ "celsius" ]
}


Renaming `celsius` touches two of the four constraints. `Symbolics.free` answers the other usual question, which
variables a constraint needs, and knows from Basic's dialect that a quantifier's `c` is bound inside it:

In [4]:
console.log(Object.fromEntries(Object.entries(constraints).map(([name, constraint]) => [name, [...Symbolics.free(constraint, E.DIALECT)].sort()])));

{
  in_range: [ "shipment" ],
  on_time: [ "shipment", "to_go" ],
  allowed: [ "shipment" ],
  reported: [ "shipment" ]
}


## Step 3: fold, bottom up

`Terms.fold(constraint, f)` calls `f(term, results)` once per term, children first, where `results` are what `f` gave
for the term's arguments. Anything computed from the parts is a fold. Here's the toolkit's `show`, in short:

In [5]:
function render(term: any, args: string[]): string {
  const form = Terms.form(term);
  const attribute = (key: string): any => form.attributes.get(key);
  if (form.kind === "literal") {
    const value = attribute("value");
    if (typeof value === "string") return `'${value}'`;
    return typeof value === "number" && Number.isInteger(value) ? value.toFixed(1) : String(value);
  }
  if (form.kind === "variable") return attribute("name");
  if (form.kind === "quantifier") return `${attribute("quantifier")}(${args[0]}, ${attribute("name")} => ${args[1]})`;
  return `${attribute("callable")}(${args.join(", ")})`;
}

console.log(Terms.fold(constraints.allowed, render));
console.log(Terms.fold(constraints.allowed, render) === show(constraints.allowed));
console.log(Terms.fold(constraints.allowed, (_term: any, depths: number[]) => 1 + Math.max(0, ...depths))); // how deep it nests

and(le(count(get(shipment, 'readings'), c => gt(c, 8.0)), 2), le(max(get(shipment, 'readings')), 12.0))


true


5


The toolkit's version adds lets and domains, and writes a bool as Python does. TypeScript writes `2.0` as `2`, so `render` writes a float's point itself, to show what Python shows.

## Step 4: same

`Terms.same(a, b)` compares two constraints by structure: the same kinds, attributes and arguments, all the way down.
Two constraints written separately are the same when they say the same thing the same way:

In [6]:
const [again, swapped] = store.transaction(() => [
  Expression((e: any) => e.root(E.literal(2.0).le(shipment.celsius).and_(shipment.celsius.le(8.0)))),
  Expression((e: any) => e.root(shipment.celsius.ge(2.0).and_(shipment.celsius.le(8.0))))]);
console.log(Terms.same(constraints.in_range, again), Terms.same(constraints.in_range, swapped));

true false


## Step 5: rewriting

`2.0 <= celsius` and `celsius >= 2.0` mean the same, and reviews would be easier if every constraint used the second. A
**rewriting** is a list of rules, each pairing a pattern with its replacement. Patterns have **holes**, which match any
term (`A`) or any attribute value (`K`, `N`), and a rule is applied wherever its pattern matches. The most specific
pattern wins, so the general rules say what stays as it is, and the specific one what changes. A pattern takes its
attributes first, then its arguments:

In [7]:
const [A, B, K, N] = [new Hole("A"), new Hole("B"), new Hole("K"), new Hole("N", String)];
const keep = [new Rule(new Pattern("variable", { name: N }), new Pattern("variable", { name: N }), "forward"),
  new Rule(new Pattern("literal", { value: K }), new Pattern("literal", { value: K }), "forward"),
  new Rule(new Pattern("call", { callable: N }, A), new Pattern("call", { callable: N }, A), "forward"),
  new Rule(new Pattern("call", { callable: N }, A, B), new Pattern("call", { callable: N }, A, B), "forward")];
const flip = [["le", "ge"], ["lt", "gt"], ["ge", "le"], ["gt", "lt"]].map(([op, mirror]) =>
  new Rule(new Pattern("call", { callable: op }, new Pattern("literal", { value: K }), A),
    new Pattern("call", { callable: mirror }, A, new Pattern("literal", { value: K })), "forward"));
const property_first = new Pairwise(E.DIALECT, E.DIALECT, [...keep, ...flip]);
const rewritten = store.transaction(() => Expression((e: any) => e.root(property_first.forward(constraints.in_range))));
console.log(show(rewritten));
console.log(Terms.same(rewritten, swapped));

and(ge(get(shipment, 'celsius'), 2.0), le(get(shipment, 'celsius'), 8.0))


true


`forward` gives a writer of the new constraint, written here as a new expression, and leaves the old one as it was:
a rewriting never changes an expression in place. The rewritten constraint is now the same as the one written property
first.

The rewriting above handles terms with one or two arguments; `allowed` has a quantifier, which it has no rule for, and
it says so rather than guessing:

In [8]:
try {
  property_first.forward(constraints.allowed);
} catch (error) {
  console.log((error as Error).message);
}

Basic quantifier 'count' has no Basic counterpart


## Why it works this way

- **A small tree, the same in every dialect.** Five kinds of term, each with a form, so an analysis is a loop or a fold,
  and it works for any dialect's expressions (case study 7), which have forms too.
- **Rewriting is translation.** A rewriting is a translator from a dialect to itself, written with the same rules as
  the translators between languages, which case study 7 uses.
- **Refuse what isn't covered.** A rewriting that met a term it had no rule for and copied it silently could change
  what a constraint means without anyone noticing.

<!-- nav -->
[← 5 · Deciding what you can now](05_Deciding_What_You_Can_Now.ipynb) · [Home](../../README.md) · [7 · One constraint, many languages →](07_One_Constraint_Many_Languages.ipynb)